In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Load the dataset
file_path = "Games22final.csv"
df = pd.read_csv(file_path)

# Ensure team name column is retained
if "team" in df.columns:
    team_names = df["team"]
else:
    raise ValueError("Error: 'team' column not found in dataset.")

# Initialize scalers
robust_scaler = RobustScaler()
minmax_scaler = MinMaxScaler()

scaled_data = {"team": team_names}  # Store team names

# Scaling numerical columns
for col in df.columns:
    if df[col].dtype in [np.int64, np.float64]:  # Only process numerical columns
        col_data = df[col].to_numpy().reshape(-1, 1)

        # Apply robust scaling
        robust_scaled = robust_scaler.fit_transform(col_data)

        # Apply min-max scaling
        final_scaled = minmax_scaler.fit_transform(robust_scaled)

        # Store flattened values in a dictionary
        scaled_data[f'{col}_final_scaled'] = final_scaled.flatten()

# Creating the final scaled DataFrame
final_df = pd.DataFrame(scaled_data)

# Handle missing values (drop or fill with mean)
final_df.dropna(inplace=True)  # Or use final_df.fillna(final_df.mean(), inplace=True)

# Save the processed data
final_csv_path = "Games22_Scaled_Weighted_100_done.csv"
final_df.to_csv(final_csv_path, index=False)

# Define the weighting function
def weighting(features, target, file_path):
    # Load the preprocessed dataset
    data = pd.read_csv(file_path)

    # Handle missing values in the dataset
    data.dropna(inplace=True)

    # Ensure features exist in the dataset
    missing_features = [feat for feat in features if feat not in data.columns]
    if missing_features:
        return f"Error: The following features are missing in the dataset: {missing_features}"

    if target not in data.columns:
        return f"Error: Target variable '{target}' not found in the dataset."

    # Splitting features and target
    X = data[features]
    y = data[target]

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train the RandomForestRegressor
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)

    # Compute feature importance
    feature_importance_dict = dict(zip(features, rf.feature_importances_))

    return feature_importance_dict

# Define feature groups and targets
offense_feats = [
    "FGR_2_final_scaled",
    "FGR_3_final_scaled",
    "FTR_final_scaled",
    "AST_final_scaled",
    "largest_lead_final_scaled",
    "TOV_final_scaled",  # TOV will be modified separately (1 - scaled value)
    "OREB_final_scaled",
    "DREB_final_scaled"
]
tar_o = "team_score_final_scaled"

defense_feats = [
    "DREB_final_scaled",
    "BLK_final_scaled",
    "STL_final_scaled",
    "F_tech_final_scaled",  # 🔴 Negative impact for Technical Fouls
    "F_personal_final_scaled"  # 🔴 Negative impact for Personal Fouls
]
tar_d = "opponent_team_score_final_scaled"

ext_feats = [
    "rest_days_final_scaled",
    "OT_length_min_tot_final_scaled",
    "tz_dif_H_E_final_scaled",
    "prev_game_dist_final_scaled",
    "home_away_NS_final_scaled",
    "travel_dist_final_scaled"
]
tar_ext = "team_score_final_scaled"

# Handle missing values before train-test split
for col in offense_feats + defense_feats + ext_feats:
    if col in final_df.columns:
        final_df[col].fillna(final_df[col].mean(), inplace=True)

# Run weighting functions for each feature group
offense_results = weighting(offense_feats, tar_o, final_csv_path)
defense_results = weighting(defense_feats, tar_d, final_csv_path)
ext_results = weighting(ext_feats, tar_ext, final_csv_path)

# Function to compute ratings
def make_rtg(final_df, weights):
    modified_values = []

    missing_features = [col for col in weights.keys() if col not in final_df.columns]
    if missing_features:
        raise ValueError(f"Error: Missing features in dataset: {missing_features}")

    for row in final_df.itertuples(index=False):
        weighted_sum = 0

        for col in weights.keys():
            if col in final_df.columns:
                if col == "TOV_final_scaled":  # Invert TOV impact
                    new_value = (1 - getattr(row, col)) * weights[col]
                elif col in ["F_tech_final_scaled", "F_personal_final_scaled"]:  # Negative impact fouls
                    new_value = -getattr(row, col) * weights[col]
                else:
                    new_value = getattr(row, col) * weights[col]

                weighted_sum += new_value

        modified_values.append(weighted_sum)

    # Convert list to DataFrame
    modified_df = pd.DataFrame({"weighted_sum": modified_values})

    # Scale weighted sum to range (0,100)
    scaled_rtg = MinMaxScaler(feature_range=(0, 100)).fit_transform(modified_df[['weighted_sum']])
    modified_df["weighted_sum_scaled"] = scaled_rtg.flatten()

    return modified_df[["weighted_sum_scaled"]]  # Return only the scaled weighted sum column

# Compute ratings
off_rtg = make_rtg(final_df, offense_results)
def_rtg = make_rtg(final_df, defense_results)
ext_rtg = make_rtg(final_df, ext_results)

# Add ratings to the final DataFrame
final_df["offense_rating"] = off_rtg.values
final_df["defense_rating"] = def_rtg.values
final_df["extra_rating"] = ext_rtg.values

# Save final DataFrame with ratings, including team names
final_rtg_csv_path = "Games22_with_Ratings-2.csv"
final_df.to_csv(final_rtg_csv_path, index=False)

# Print confirmation
print(f"Final dataset with ratings saved as {final_rtg_csv_path}")

# Attempt download (only works in Google Colab)
try:
    from google.colab import files
    files.download(final_rtg_csv_path)
    print("Download started for:", final_rtg_csv_path)
except ImportError:
    print("Not running in Google Colab, skipping download.")

<ipython-input-12-f0674c0c628f>:114: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  final_df[col].fillna(final_df[col].mean(), inplace=True)


Final dataset with ratings saved as Games22_with_Ratings-2.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started for: Games22_with_Ratings-2.csv
